# Discovery Engine — Blinkit Review Analysis (Notebook Fallback)

Stage 10 (`architecture.md` §4) fallback presentation path — the primary UI is `app.py`
(`streamlit run app.py`); this notebook reads the exact same artifacts
(`data/insights.json`, `data/themes.json`, `data/validation.json`, plus
`data/reviews.jsonl`/`data/units.jsonl` for corpus counts) and answers the same
questions in a static/portable, dependency-light form (no Streamlit required —
just pandas + stdlib), for evaluators who prefer `jupyter notebook`/`nbconvert`/
`nbviewer` over running a server.

**Read order:** run all cells top to bottom. Each section corresponds to one of the
demonstration requirements in `problemstatement.md` §4:

1. How data is gathered and analyzed (pipeline overview + corpus stats)
2. How themes are identified (theme browser)
3. How insights are generated (8 research questions, evidence-backed)
4. How insight quality was validated (coherence + triangulation + spot-check)

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(Path.cwd()))  # allow `import src.*` when run from the project root

from src.config import load_config
from src.schema import read_json

config = load_config()

insights = read_json(config.paths.insights)
themes_doc = read_json(config.paths.themes)
validation = read_json(config.paths.validation)

themes = themes_doc["themes"]
signals = themes_doc["emerging_signals"]
themes_by_id = {t["theme_id"]: t for t in themes}
signals_by_id = {s["signal_id"]: s for s in signals}

print(f"Loaded {len(themes)} themes, {len(signals)} emerging signals, "
      f"{len(insights['questions'])} research questions, "
      f"{len(validation['coherence']['themes'])} validated themes.")

Loaded 40 themes, 819 emerging signals, 8 research questions, 40 validated themes.


## 1. How data is gathered and analyzed

Pipeline stages S1→S9 (each an independently re-runnable `data/` artifact — see
`architecture.md` §3), plus live corpus statistics computed from the actual
`reviews.jsonl`/`units.jsonl` on disk (not hardcoded).

In [2]:
STAGE_TABLE = [
    ("S1", "Scrape", "src/scrape.py", "raw_reviews.jsonl", "Rolling 4-month Play Store pull, all rating bands"),
    ("S2", "Normalize", "src/normalize.py", "reviews.jsonl", "Canonical schema, ISO dates, dedup/lang tagging"),
    ("S3", "Unit extraction", "src/units.py", "units.jsonl", "Rule-based split into atomic complaint/insight statements"),
    ("S4", "Embed", "src/embed.py", "embeddings.npy", "Local all-MiniLM-L6-v2 sentence embeddings"),
    ("S5", "Similarity graph", "src/graph.py", "graph.gpickle", "kNN cosine graph over unit embeddings"),
    ("S6", "Community detection", "src/cluster.py", "communities.json", "Louvain clustering into theme communities"),
    ("S7", "Summarization", "src/summarize.py", "themes.json", "Label + description + verbatims per community"),
    ("S8", "Insight mapping", "src/insights.py", "insights.json", "Themes mapped to the 8 research questions"),
    ("S9", "Validation", "src/validate.py", "validation.json", "Coherence, triangulation, spot-check sample"),
]
display(pd.DataFrame(STAGE_TABLE, columns=["Stage", "Name", "Module", "Artifact", "What it does"]))


def _iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


rating_counts, lang_counts, dates = Counter(), Counter(), []
num_reviews = 0
for row in _iter_jsonl(config.paths.reviews):
    num_reviews += 1
    if row.get("rating") is not None:
        rating_counts[int(row["rating"])] += 1
    lang_counts[(row.get("metadata") or {}).get("lang", "en")] += 1
    if row.get("date"):
        dates.append(row["date"])
num_units = sum(1 for _ in _iter_jsonl(config.paths.units))

print(f"Reviews: {num_reviews:,}  |  Units: {num_units:,}  |  Window: {min(dates)} -> {max(dates)}")
print(f"Rating distribution: {dict(sorted(rating_counts.items()))}")
print(f"Language split: {dict(lang_counts)}")

,Stage,Name,Module,Artifact,What it does
0,S1,Scrape,src/scrape.py,raw_reviews.jsonl,"Rolling 4-month Play Store pull, all rating bands"
1,S2,Normalize,src/normalize.py,reviews.jsonl,"Canonical schema, ISO dates, dedup/lang tagging"
2,S3,Unit extraction,src/units.py,units.jsonl,Rule-based split into atomic complaint/insight...
3,S4,Embed,src/embed.py,embeddings.npy,Local all-MiniLM-L6-v2 sentence embeddings
4,S5,Similarity graph,src/graph.py,graph.gpickle,kNN cosine graph over unit embeddings
5,S6,Community detection,src/cluster.py,communities.json,Louvain clustering into theme communities
6,S7,Summarization,src/summarize.py,themes.json,Label + description + verbatims per community
7,S8,Insight mapping,src/insights.py,insights.json,Themes mapped to the 8 research questions
8,S9,Validation,src/validate.py,validation.json,"Coherence, triangulation, spot-check sample"


Reviews: 156,219  |  Units: 89,295  |  Window: 2026-03-22T00:00:28Z -> 2026-07-22T00:19:04Z
Rating distribution: {1: 27181, 2: 4191, 3: 7096, 4: 17751, 5: 100000}
Language split: {'en': 155647, 'hi': 572}


## 2. How themes are identified

Bottom-up: kNN similarity graph over unit embeddings (S5) → Louvain community
detection (S6) → local summarization into a label + description + verbatim quotes
per community (S7). The 8 research questions are **never** an input to this step —
mapping to them happens strictly after the fact (Section 3 below).

Top themes by size, and the full theme table (sortable/filterable in pandas):

In [3]:
# Reverse insights.json's question->theme_ids into theme_id->question_ids
# (themes.json's own `questions` field is never populated by Stage 8).
theme_to_q, signal_to_q = {}, {}
for q in insights["questions"]:
    for tid in q["theme_ids"]:
        theme_to_q.setdefault(tid, []).append(q["question_id"])
    for sid in q["signal_ids"]:
        signal_to_q.setdefault(sid, []).append(q["question_id"])

themes_df = pd.DataFrame(
    [
        {
            "theme_id": t["theme_id"],
            "label": t["label"],
            "member_count": t["member_count"],
            "sentiment": t["sentiment"],
            "questions": ", ".join(f"Q{q}" for q in sorted(theme_to_q.get(t["theme_id"], []))) or "uncategorized",
        }
        for t in themes
    ]
).sort_values("member_count", ascending=False)

display(themes_df.head(10))
print(f"\n... {len(themes_df)} themes total, {len(signals)} long-tail emerging signals (below "
      f"clustering.min_community_size, kept as low-confidence supplementary evidence).")

,theme_id,label,member_count,sentiment,questions
0,theme-0000,blinkit / good / app / service,8609,neutral,uncategorized
1,theme-0001,hai / app / delivery / order,5707,neutral,Q1
2,theme-0002,customer / service / support / care,5515,negative,Q6
3,theme-0003,delivery / service / location / boy,5339,negative,"Q1, Q6"
4,theme-0004,app / good / best / nice,5076,positive,uncategorized
5,theme-0005,delivery / fast / good / time,4970,positive,Q1
6,theme-0006,charges / delivery / charge / high,4851,negative,"Q1, Q6"
7,theme-0007,time / delivery / late / order,4304,neutral,Q1
8,theme-0008,service / good / fast / nice,4061,positive,Q1
9,theme-0009,vegetables / items / expired / ordered,3877,negative,Q1



... 40 themes total, 819 long-tail emerging signals (below clustering.min_community_size, kept as low-confidence supplementary evidence).


In [4]:
# Inspect any single theme by id (edit THEME_ID and re-run this cell):
THEME_ID = themes_df.iloc[0]["theme_id"]
theme = themes_by_id[THEME_ID]

display(Markdown(f"### {theme['theme_id']} — {theme['label']}"))
print(theme["description"])
print(f"Members: {theme['member_count']}  |  Sentiment: {theme['sentiment']}  |  "
      f"Questions: {theme_to_q.get(THEME_ID, [])}")
print("\nRepresentative verbatims:")
for quote in theme["representative_quotes"]:
    print(f"  - {quote}")

### theme-0000 — blinkit / good / app / service

8609 reviews (avg rating 3.2) recurring around: blinkit, good, app, service.
Members: 8609  |  Sentiment: neutral  |  Questions: []

Representative verbatims:
  - Nice to shop with Blinkit
  - Blinkit is better than before, good service 👏
  - having very good experience with blinkit


## 3. How insights are generated — the 8 research questions, evidence-backed

Each theme/signal is mapped to a question by embedding-similarity between its own
`label + description` and a short topic description per question
(`config.yaml: insights.question_queries`) — no keywords, no LLM call. `coverage`
is `"sufficient"` only when at least one real theme (not just a low-confidence
emerging signal) clears the similarity threshold — reported honestly rather than
force-matched (edgecases.md S8-01).

In [5]:
RESEARCH_QUESTIONS = {
    1: "Why do users repeatedly buy from the same categories?",
    2: "What prevents users from exploring new categories?",
    3: "How do users discover products today?",
    4: "What role do habits play in shopping behavior?",
    5: "What information do users need before trying a new category?",
    6: "What frustrations emerge repeatedly?",
    7: "Which user segments are more likely to experiment?",
    8: "What unmet needs emerge consistently across discussions?",
}

overview_df = pd.DataFrame(
    [
        {
            "Q": q["question_id"],
            "Question": RESEARCH_QUESTIONS[q["question_id"]],
            "Coverage": q["coverage"],
            "Themes": len(q["theme_ids"]),
            "Unit count": q["total_count"],
            "Signals (weak evidence)": len(q["signal_ids"]),
        }
        for q in insights["questions"]
    ]
)
display(overview_df)

,Q,Question,Coverage,Themes,Unit count,Signals (weak evidence)
0,1,Why do users repeatedly buy from the same cate...,sufficient,19,55086,7
1,2,What prevents users from exploring new categor...,insufficient,0,0,0
2,3,How do users discover products today?,sufficient,2,2341,3
3,4,What role do habits play in shopping behavior?,sufficient,1,2056,0
4,5,What information do users need before trying a...,insufficient,0,0,4
5,6,What frustrations emerge repeatedly?,sufficient,11,30448,28
6,7,Which user segments are more likely to experim...,insufficient,0,0,1
7,8,What unmet needs emerge consistently across di...,insufficient,0,0,0


In [6]:
def show_question(question_id: int) -> None:
    q = next(q for q in insights["questions"] if q["question_id"] == question_id)
    display(Markdown(f"### Q{question_id}: {RESEARCH_QUESTIONS[question_id]}"))
    print(f"Embedding query used for matching: \u201c{q['query']}\u201d")
    print(f"Coverage: {q['coverage']}  |  Supporting themes: {len(q['theme_ids'])}  |  "
          f"Unit count: {q['total_count']}  |  Signal support: {q['signal_support_total']}")

    if q["theme_ids"]:
        rows = [
            {
                "theme_id": tid,
                "label": themes_by_id[tid]["label"],
                "similarity": q["theme_similarities"].get(tid),
                "member_count": themes_by_id[tid]["member_count"],
                "sentiment": themes_by_id[tid]["sentiment"],
            }
            for tid in q["theme_ids"]
            if tid in themes_by_id
        ]
        display(pd.DataFrame(rows))
    else:
        print("(No theme clears the similarity threshold for this question yet.)")

    if q["top_verbatims"]:
        print("\nRepresentative verbatims:")
        for v in q["top_verbatims"]:
            print(f"  - {v}")


# Show all 8 questions end to end:
for qid in range(1, 9):
    show_question(qid)
    print("\n" + "-" * 100 + "\n")

### Q1: Why do users repeatedly buy from the same categories?

Embedding query used for matching: “Repeat purchases from the same familiar categories; brand loyalty; routine reordering habits”
Coverage: sufficient  |  Supporting themes: 19  |  Unit count: 55086  |  Signal support: 7


,theme_id,label,similarity,member_count,sentiment
0,theme-0018,app / best / good / shopping,0.444,2056,positive
1,theme-0031,stock / items / products / things,0.429,285,neutral
2,theme-0010,order / ordered / items / delivered,0.419,3758,negative
3,theme-0011,price / good / quality / product,0.373,3576,neutral
4,theme-0007,time / delivery / late / order,0.367,4304,neutral
5,theme-0034,tip / tips / delivery / option,0.367,88,negative
6,theme-0027,blinkit / app / order / good,0.348,402,neutral
7,theme-0022,discount / offers / offer / coupon,0.339,916,neutral
8,theme-0006,charges / delivery / charge / high,0.332,4851,negative
9,theme-0015,app / delivery / fast / best,0.328,2871,positive



Representative verbatims:
  - NICE APP FOR GROCERIES AND OTHER PRODUCTS
  - good app and all groceries products are available
  - good and best app for groceries
  - most of items is out of stock
  - most of the items are out of stock always

----------------------------------------------------------------------------------------------------



### Q2: What prevents users from exploring new categories?

...


Representative verbatims:
  - this is the worst app I have ever seen i suggest you all not to order anything from this app they are biggest online farauders I have ordered something and then I returned it
  - dont order this app very bad app
  - This App Is Very Bad They Sell very bad product
  - one time my order has not came
  - Will not give any order next time

----------------------------------------------------------------------------------------------------



### Q7: Which user segments are more likely to experiment?

Embedding query used for matching: “User segments more open to trying new things versus sticking to routine”
Coverage: insufficient  |  Supporting themes: 0  |  Unit count: 0  |  Signal support: 2
(No theme clears the similarity threshold for this question yet.)

----------------------------------------------------------------------------------------------------



### Q8: What unmet needs emerge consistently across discussions?

Embedding query used for matching: “Unmet needs, missing features, or product gaps mentioned repeatedly”
Coverage: insufficient  |  Supporting themes: 0  |  Unit count: 0  |  Signal support: 0
(No theme clears the similarity threshold for this question yet.)

----------------------------------------------------------------------------------------------------



### Category graph

A theme-to-theme similarity map (problemstatement.md §7) — each theme's centroid
embedding compared against every other theme's; top-5 most similar neighbors kept
as edges. Shown here as a plain edge table (the interactive force-directed version
lives in `app.py`'s "Category Graph" tab).

In [7]:
category_graph_df = pd.DataFrame(insights["category_graph"]).sort_values("similarity", ascending=False)
category_graph_df["label_a"] = category_graph_df["theme_a"].map(lambda t: themes_by_id[t]["label"])
category_graph_df["label_b"] = category_graph_df["theme_b"].map(lambda t: themes_by_id[t]["label"])
display(category_graph_df.head(15))

,theme_a,theme_b,similarity,label_a,label_b
70,theme-0014,theme-0004,0.834,app / worst / use / bad,app / good / best / nice
20,theme-0004,theme-0014,0.834,app / good / best / nice,app / worst / use / bad
75,theme-0015,theme-0018,0.787,app / delivery / fast / best,app / best / good / shopping
90,theme-0018,theme-0015,0.787,app / best / good / shopping,app / delivery / fast / best
76,theme-0015,theme-0014,0.782,app / delivery / fast / best,app / worst / use / bad
71,theme-0014,theme-0015,0.782,app / worst / use / bad,app / delivery / fast / best
21,theme-0004,theme-0015,0.772,app / good / best / nice,app / delivery / fast / best
77,theme-0015,theme-0004,0.772,app / delivery / fast / best,app / good / best / nice
25,theme-0005,theme-0007,0.765,delivery / fast / good / time,time / delivery / late / order
35,theme-0007,theme-0005,0.765,time / delivery / late / order,delivery / fast / good / time


## 4. How insight quality was validated

Three independent checks over the 40 qualifying themes (Stage 9, `src/validate.py`):

1. **Coherence** — graph modularity of the actual Louvain partition (primary,
   methodologically-correct metric for a graph-clustering method) + a per-theme
   centroid-based silhouette-style score.
2. **Cross-segment triangulation** — does each theme hold up across rating bands,
   time cohorts, and review-length buckets, or is it an artifact of one narrow slice?
3. **Human spot-check** — a stratified random sample of member units per theme,
   exported for manual labeling (`data/spot_check_sample.json`).

In [8]:
v = validation["summary"]
print(f"Themes validated: {v['num_themes_validated']}")
print(f"Graph modularity: {v['modularity']}")
print(f"Mean silhouette-style score: {v['mean_silhouette_score']}")
print(f"Cross-segment stable: {v['num_cross_segment_stable']}/{v['num_themes_validated']}")
print(f"Segment-specific: {v['num_segment_specific']}/{v['num_themes_validated']}")
print(f"Spot-check labels collected so far: {v['spot_check_labeled_count']} "
      f"(agreement rate: {v['spot_check_agreement_rate']})")

coherence_df = pd.DataFrame(validation["coherence"]["themes"])
coherence_df["label"] = coherence_df["theme_id"].map(lambda t: themes_by_id.get(t, {}).get("label", ""))
display(coherence_df.sort_values("silhouette_score", na_position="last")[
    ["theme_id", "label", "size", "intra_similarity", "nearest_theme_id", "nearest_theme_similarity", "silhouette_score"]
])

Themes validated: 40
Graph modularity: 0.8394
Mean silhouette-style score: -0.001
Cross-segment stable: 28/40
Segment-specific: 12/40
Spot-check labels collected so far: 0 (agreement rate: None)


,theme_id,label,size,intra_similarity,nearest_theme_id,nearest_theme_similarity,silhouette_score
9,theme-0009,vegetables / items / expired / ordered,3877,0.490,theme-0010,0.741,-0.339
3,theme-0003,delivery / service / location / boy,5339,0.499,theme-0005,0.749,-0.334
2,theme-0002,customer / service / support / care,5515,0.511,theme-0003,0.747,-0.316
13,theme-0013,fast / good / helpful / easy,3193,0.506,theme-0008,0.723,-0.300
10,theme-0010,order / ordered / items / delivered,3758,0.544,theme-0009,0.741,-0.267
14,theme-0014,app / worst / use / bad,2892,0.627,theme-0004,0.834,-0.249
17,theme-0017,good / nice / best / like,2309,0.515,theme-0013,0.672,-0.234
18,theme-0018,app / best / good / shopping,2056,0.612,theme-0015,0.787,-0.223
11,theme-0011,price / good / quality / product,3576,0.526,theme-0009,0.674,-0.219
24,theme-0024,app / minute / best / good,729,0.590,theme-0004,0.743,-0.206


In [9]:
tri_df = pd.DataFrame(validation["triangulation"]["themes"])
print("Stability counts:", Counter(tri_df["stability"]).most_common())

specific = tri_df[tri_df["stability"] == "segment_specific"]
specific = specific.assign(label=specific["theme_id"].map(lambda t: themes_by_id.get(t, {}).get("label", "")))
display(specific[["theme_id", "label", "stability_reasons"]])

print("\n" + validation["spot_check"]["note"])

Stability counts: [('cross_segment', 28), ('segment_specific', 12)]


,theme_id,label,stability_reasons
4,theme-0004,app / good / best / nice,[concentrated in a single review-length bucket]
8,theme-0008,service / good / fast / nice,[concentrated in a single review-length bucket]
13,theme-0013,fast / good / helpful / easy,[concentrated in a single review-length bucket]
17,theme-0017,good / nice / best / like,[concentrated in a single review-length bucket]
19,theme-0019,experience / good / bad / worst,[concentrated in a single review-length bucket]
32,theme-0032,good / work / great / job,[concentrated in a single review-length bucket]
33,theme-0033,thank / blinkit / thanks / time,[concentrated in a single review-length bucket]
35,theme-0035,expansive / bit / compare / big,[concentrated in a single review-length bucket]
36,theme-0036,joke / joking / right,[concentrated in a single review-length bucket]
37,theme-0037,watch / worth / ordered / titan,"[concentrated in a single time cohort, concent..."



No human labels present yet - open data/spot_check_sample.json, fill in `human_agrees` (true/false) per row, then re-run `python -m src.validate --refresh` to compute agreement (S9-04).
